##### Copyright 2026 Google LLC.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Gemini API: Get started with Lyria 3 🎵

This notebook shows you how to generate high-fidelity music with **Lyria 3**, Google DeepMind's family of music generation models.

Whether you need a short 30-second jingle or a full-length 4-minute song with vocals and structural complexity, Lyria 3 makes it possible with the same simple `generate_content` call you're already familiar with.

In this notebook, you will learn how to:
- Generate music from text prompts.
- Create "visual soundtracks" where an image serves as the inspiration.
- Use advanced timestamped prompts to control song structure.
- Generate purely instrumental tracks.
- Create vocal tracks in multiple languages.

### Setup
Ensure you have the latest version of the Google Gen AI Python SDK.

```bash
pip install -U "google-genai>=1.62.0"
```

In [ ]:
%pip install -U -q "google-genai>=1.62.0"

### Setup your API key
You can get an API key from [Google AI Studio](https://aistudio.google.com/apikey).

In a Colab, use your secret called `GOOGLE_API_KEY`:

In [ ]:
from google import genai
from google.genai import types

# Access your API key (ensure it's in Colab secrets)
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

client = genai.Client(api_key=GOOGLE_API_KEY)

# List all Lyria models
for model in client.models.list():
    if 'lyria' in model.name:
        print(f"Model: {model.name}")

## Generate your first song

Generating a song is very simple as it's using the same `generate_content` call you're already familiar with. You MUST specify the `Audio` and `Text` modalities in the configuration to receive the audio data and the generated lyrics/structure.

The 30-second model (`lyria-3-clip-preview`) is best for short clips and loops. The full-song model (`lyria-3-pro-preview`) handles longer durations and carries a more complex emotional arc. Let's start with a 30-second clip:

In [ ]:
model_id = "lyria-3-clip-preview"

prompt = "Create a 30-second cheerful acoustic folk song about a sunrise in the mountains."

response = client.models.generate_content(
    model=model_id,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['Audio', 'Text']
    )
)

# Lyrics/structure
print(f"Lyrics:\n{response.parts[0].text}")

# High-fidelity audio (48kHz stereo)
for part in response.parts:
    if part.inline_data:
        from IPython.display import Audio, display
        display(Audio(part.inline_data.data, rate=48000))

## Visual inspiration: Image-to-Music 🖼️➡️🎵

Lyria 3 can "invent" a song from a visual cue. Passing an image in the `contents` list gives the model a whole new dimension of creativity. Let's see how a sunset in the desert can inspire a song.

First, download a test image:

In [ ]:
from PIL import Image
import requests
from io import BytesIO

# Download an example image
url = "https://upload.wikimedia.org/wikipedia/commons/e/ec/Grand_Canyon_with_sunset.jpg"
image_response = requests.get(url)
image = Image.open(BytesIO(image_response.content))

# Generate music inspired by the image
response = client.models.generate_content(
    model="lyria-3-pro-preview",
    contents=[
        "An atmospheric ambient track inspired by the mood and colors in this image. Deep synths and a slow tempo.",
        image
    ],
    config=types.GenerateContentConfig(
        response_modalities=['Audio', 'Text'],
    ),
)
# Display image and audio
display(image)
for part in response.parts:
    if part.inline_data:
        display(Audio(part.inline_data.data, rate=48000))

## Advanced prompting: Timing and structure ⏱️🎼

You can specify exactly what happens at specific moments in the song using timestamps. This is useful for distributing lyrics or controlling instrumental shifts as the composition develops.

Let's tell the model what to do in each 15-second block of a song:

In [ ]:
prompt = """
[0:00 - 0:15] Intro: Fast acoustic guitar arpeggios, setting an energetic tone.
[0:15 - 0:45] Verse 1: Add a warm Fender Rhodes piano melody.
[0:45 - 1:00] Chorus: Full band with upbeat drums and soaring synth leads.
"""

response = client.models.generate_content(
    model="lyria-3-pro-preview",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['Audio', 'Text']
    )
)

print(f"Lyrics:\n{response.parts[0].text}")

for part in response.parts:
    if part.inline_data:
        display(Audio(part.inline_data.data, rate=48000))

## Instrumental-only generation 🎹

For background music, soundtracks, or game loops where vocals aren't needed, you can use the `instrumental_only` parameter. This is faster than adding "instrumental" to your prompt and ensures a vocal-free track.

## Multilingual music 🌍🎤

Lyria 3 generates lyrics in the language it sees in your prompt. By providing instructions in French, Spanish, or Japanese, the model will produce a vocal performance that matches the pronunciation and emotional tone of that language. Let's create a disco track in Spanish:

In [ ]:
prompt = "Crea una canción de música disco alegre sobre un viaje a la luna."

response = client.models.generate_content(
    model="lyria-3-clip-preview",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['Audio', 'Text'],
    ),
)

# Lyrics in Spanish
print(f"Lyrics (Spanish):\n{response.parts[0].text}")

for part in response.parts:
    if part.inline_data:
        display(Audio(part.inline_data.data, rate=48000))

## What's next?

- Listen to your creations! 🎧
- Explore [Google AI Studio](https://aistudio.google.com) to iterate on your prompts visually.
- Combine music generation with [image generation](https://ai.google.dev/gemini-api/docs/image-generation) for full multimedia content.
- Read the [Gemini API reference](https://ai.google.dev/api) for more details on `GenerateContentConfig`.
- Dive into the [Lyria 3 technical report](https://deepmind.google/technologies/lyria/) to learn about the architecture behind these models.